

*   **--------------------------------------------Name: Vanchanagiri Alekhya-----------------------------------------**
*   **----------------------------------------Enrollment Number: MML2025002----------------------------------------**

*   **------------------------------------------GenAI & LLMs Assignment-3
--------------------------------------------**


**PART - 1**
---

In industry, embedding for recommendation system is widely used.
Use word2vec algorithm to embed songs and use human made music playlist and then use these embeddings to recommend similar songs that often appear together in the playlist. Treat each song as a word or token and playlist like a sentence.

[hints: Use Cornel University dataset ( https://oreil.ly/A-AK6), train a Song Embedding Model and use those embeddings to find similar songs. Ref Chapter -2 of our text book ].


In [1]:
!pip install gensim
import pandas as pd
import numpy as np
from urllib import request
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.0 MB/s eta 0:00:00


In [2]:
data = request.urlopen(
    'https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt'
)
lines = data.read().decode("utf-8").split('\n')[2:]
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]
songs_file = request.urlopen(
    'https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt'
)
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')
model = Word2Vec(
    vector_size=32,
    window=20,
    negative=50,
    min_count=1,
    workers=4
)

model.build_vocab(playlists)
model.train(
    playlists,
    total_examples=model.corpus_count,
    epochs=10
)

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id), topn=5)
    )[:, 0]
    return songs_df.iloc[similar_songs]



In [3]:
print(print_recommendations(3167))

                    title           artist
id                                        
6622      Smooth Up In Ya       Bulletboys
6637            Seventeen           Winger
1785    Flying High Again    Ozzy Osbourne
11555       House Of Pain  Faster Pussycat
33743        Blow My Fuse              Kix


**PART - 2**
---

**Q1. Tokenization & Meaning (10 Marks)**
---

**Explain the following in your own words:**


**a) Why do LLMs not operate directly on words?**

ans:  

Large Language Models do not operate directly on words because word-level representations are inefficient and inflexible.

Natural language has an extremely large and evolving vocabulary, which would require massive embedding tables if every distinct word were treated as a separate token.

Word-level models also struggle with unseen or rare words, leading to out-of-vocabulary (OOV) problems.

Additionally, words do not naturally capture internal structure such as prefixes, suffixes, or roots.

LLMs therefore operate on tokens, which are usually subword units, allowing them to model language more compactly and generalize better.

**b) How subword tokenization helps with:**

**•	Rare words:**

ans:

Subword tokenization breaks rare or unseen words into smaller, known pieces. Instead of treating an unknown word as entirely new, the model represents it using familiar subword units, enabling it to infer meaning from components it has already learned.

**•	morphology:**

ans:

Subword tokens capture meaningful word parts such as prefixes (un-, re-), suffixes (-ing, -tion), and roots. This helps models understand grammatical variations and relationships between related words like run, running, and runner.

**•	open vocabulary**

ans:

With subword tokenization, a model can represent any new word by decomposing it into subword units, even if the word never appeared during training. This removes the fixed vocabulary limitation and allows LLMs to handle new names, technical terms, and evolving language.


**c) Give one example where word-level tokenization fails but subword tokenization works well.**

ans:

Example-1: “unhappiness”

**Word-level tokenization:**

The entire word unhappiness may be treated as an unknown or rare token, so the model fails to capture its meaning.

**Subword tokenization:** The word is split into meaningful parts such as un + happy + ness, allowing the model to understand the negation, root meaning, and noun form.

This is because Subword tokenization preserves semantic and morphological information, enabling the model to generalize even when the full word was not seen during training.

-------
Example-2: “biodegradable”

**Word-level tokenization:**

May be unknown or poorly represented.

**Subword tokenization:** bio + degrade + able

This gives environmental and process-related meaning.

**Q2. Embedding Space Intuition (10 Marks)**
---



**Consider the words:**

**king, queen, man, woman, doctor, nurse**



**a) What does it mean to say these words lie in a “vector space”?**

ans:  

Saying that words like king, queen, man, woman, doctor, and nurse lie in a vector space means that each word is represented as a numerical vector in a high-dimensional space.

The position of each word is learned from data such that words with similar meanings or usage contexts are located closer together.

Relationships between words can be captured through geometric operations, for example, the relationship between king and queen reflects gender differences similar to the relationship between man and woman.



**b) Why is cosine similarity preferred over Euclidean distance in embedding spaces?**

ans:  

Cosine similarity is preferred because it measures the angular similarity between vectors, focusing on their direction rather than their absolute magnitude.

In embedding spaces, the direction of a vector encodes semantic meaning, while the magnitude can be influenced by factors such as word frequency or training dynamics.

Euclidean distance, on the other hand, is sensitive to vector length and can incorrectly treat vectors with the same semantic meaning but different magnitudes as dissimilar.

Additionally, in high-dimensional spaces, Euclidean distances tend to become less discriminative due to the “curse of dimensionality,” whereas cosine similarity remains stable and effective for measuring semantic relatedness.



**c) Can embeddings capture bias? Briefly explain.**

ans:

Yes, embeddings can capture and even amplify social, cultural, and gender biases present in the data used to train them.

Since embeddings learn statistical patterns from real-world text, any biases reflected in language usage—such as associating certain professions with specific genders—can become encoded in the vector space.

For example, embeddings may place doctor closer to man and nurse closer to woman if such patterns are prevalent in the training data.

These biases can influence downstream applications, leading to unfair or discriminatory outcomes, which is why bias detection and mitigation techniques are important in embedding-based systems.

**Q3. Semantic vs Lexical Similarity (10 Marks)**
---



**Explain the difference using the following sentence pairs:**

1.	“How to bake a cake?”

    “Steps for making a cake”
    
2.	“Apple released a new phone”

    “Apple is a healthy fruit”


**Which pairs will embedding similarity score higher, and why?**

ans:

Lexical similarity measures similarity based on surface-level word overlap (exact or near-exact words).

Semantic similarity measures similarity based on the underlying meaning, even if different words are used.

**Sentence Pair 1:**


“How to bake a cake?”

“Steps for making a cake”

**1. Lexical similarity:** Moderate, since both sentences share common words like cake and related cooking terms.

**2. Semantic similarity:** High, because both sentences refer to the same intent — instructions for preparing a cake.

**3. Embedding similarity score:** High, as embeddings capture meaning rather than exact word matches.

**Sentence Pair 2:**

“Apple released a new phone”

“Apple is a healthy fruit”

**1. Lexical similarity:** Moderate, because both sentences contain the word Apple.

**2. Semantic similarity:** Low, as Apple refers to a technology company in the first sentence and a fruit in the second.

**3. Embedding similarity score:** Low, because embeddings use context to disambiguate word meanings.

**Conclusion:**

 Sentence Pair 1 will have a higher embedding similarity score due to strong semantic alignment. Sentence Pair 2 will have a lower embedding similarity score despite lexical overlap.

 This shows that embeddings focus on semantic meaning rather than surface-level word similarity.

**Q4. Embedding Generation & Similarity (20 Marks)**
---


Task:

1.	Choose any pre-trained embedding model
2.	Compute embeddings for the following sentences:

S1: Robots can assist humans in daily tasks

S2: Machines help people in everyday activities

S3: The stock market crashed yesterday

3.	Compute pairwise cosine similarity

4.	Comment on the results

Expected Deliverables:

•	Code

•	Similarity matrix

•	5-6 lines interpretation


In [4]:
!pip install sentence-transformers


In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [6]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:

sentences = [
    "Robots can assist humans in daily tasks",
    "Machines help people in everyday activities",
    "The stock market crashed yesterday"
]


In [8]:
embeddings = model.encode(sentences)

In [9]:
similarity_matrix = cosine_similarity(embeddings)

In [10]:
labels = ["S1", "S2", "S3"]
similarity_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)

print(similarity_df)


          S1        S2        S3
S1  1.000000  0.678374  0.060709
S2  0.678374  1.000000  0.014423
S3  0.060709  0.014423  1.000000


**Interpretation:**
---

1. Sentences S1 and S2 have a high cosine similarity, indicating that they have strong semantic similarity.

2. Both sentences S1 and S2 describe machines assisting humans in daily activities, even though different words are used.

3. Sentence S3 has very low similarity with S1 and S2, as it discusses a completely unrelated topic (finance).

4. This shows that embedding models capture semantic meaning, not just exact word overlap.

5. Cosine similarity effectively measures how close sentence meanings are in embedding space.

6. Pre-trained embeddings can generalize well without task-specific training.

**Q5. Mini Semantic Search Engine (25 Marks)**
---



Task:

Build a tiny semantic search system using embeddings.

Corpus (minimum 8 documents):

•	Choose from any one domain:

1.	Healthcare

2.	Robotics

3.	Education

4.	News

Steps:

1.	Convert documents to embeddings

2.	Accept a user query

3.	Retrieve top-3 most similar documents

4.	Display similarity scores

In [11]:
!pip install sentence-transformers scikit-learn


In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

In [13]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
documents = [
    "Robots are widely used in manufacturing industries to automate repetitive tasks. "
    "They help reduce human effort and improve productivity.",

    "Industrial robots are designed to perform precise and efficient operations on assembly lines. "
    "They improve accuracy and reduce production time.",

    "Service robots assist humans in healthcare and domestic environments. "
    "They support tasks such as patient care and household assistance.",

    "Autonomous robots can navigate unknown environments and make decisions without human intervention. "
    "They rely on sensors and artificial intelligence.",

    "Machine learning enables robots to learn from data and past experiences. "
    "This helps them adapt to new situations and improve performance over time.",

    "Humanoid robots are built to resemble human appearance and behavior. "
    "They are often used in research and human-robot interaction studies.",

    "Robots are increasingly used in medical surgeries to achieve high accuracy. "
    "They assist doctors in performing complex procedures.",

    "Recent advances in sensors and artificial intelligence have improved robotic perception. "
    "This allows robots to better understand and interact with their environment."
]
doc_embeddings = model.encode(documents)

query = "robots helping humans in healthcare"
query_embedding = model.encode([query])
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_k = 3
top_indices = np.argsort(similarities)[-top_k:][::-1]
results = []
for idx in top_indices:
    results.append({
        "Document": documents[idx],
        "Similarity Score": round(similarities[idx], 3)
    })

results_df = pd.DataFrame(results)
print(results_df)


                                            Document  Similarity Score
0  Service robots assist humans in healthcare and...             0.793
1  Robots are increasingly used in medical surger...             0.667
2  Robots are widely used in manufacturing indust...             0.567


**Q6. Limitations of Embeddings (5 Marks)**
---
Answer briefly:



**a) Can embeddings fully capture context?**

ans:

No, since embeddings compress meaning into fixed-length vectors, they are unable to properly express complex context, including speaker intent, discourse-specific meaning, temporal information, and long-range dependencies, but they are usually able to convey general semantics.

In particular, static embeddings give a word the same vector regardless of its context, and when contextual embeddings are employed in isolation, they may lose subtle details.



**b) Why might embeddings fail for numerical reasoning?**

ans:

Embeddings do not understand mathematical relationships, magnitudes, or precise values since they encode numbers as symbols rather than quantities.

Because vector similarity doesn't correspond to numerical correctness, they thus have difficulty with tasks that require arithmetic, counting, comparisons, or logical constraints.


**c) Why do embeddings struggle with negation?**

ans:

Negation often causes subtle changes in wording but a complete reversal of meaning.

However, embedding models tend to emphasize shared words and overall semantic similarity, causing sentences like “This is good” and “This is not good” to appear close in embedding space.

This makes it difficult for embeddings to reliably distinguish opposite meanings.